# Skin change vs detection recall

The FE skin panel lets you re-texture any object class at runtime. This notebook quantifies what that does to the HSV detector and demonstrates the recovery path (`POST /profiles/<class>` resampling) — fully offline, no ROS or browser needed.

Workflow reproduced here:
1. Evaluate towel recall on synthetic eval frames with the **default** towel appearance.
2. Re-skin the towels (the blue from the FE `STRIPED` preset) and watch recall collapse to zero.
3. Resample the towel HSV profile from the skin image (exactly what the FE `sync AI profile` checkbox does) and watch recall recover.

Verified result with seed 42: **0.87 -> 0.00 -> 0.95**.

Run inside the `ai_worker` container or any env with `opencv-python` + `numpy`. Note: low-saturation skins (e.g. the `CHARCOAL` preset) have unstable hue under sensor noise — a resampled HSV band recovers them only partially; that failure mode is a good exercise for trying an ONNX detector behind the same `Detector` interface.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))

import cv2
import numpy as np
import matplotlib.pyplot as plt

from onsen_ai_worker.detection import Detector, iou

rng = np.random.default_rng(42)
detector = Detector(camera_model=None)
print('default towel band:', detector.profiles_snapshot()['towel'])

## Synthetic eval frames

Each frame is a dark wooden onsen floor (gradient + sensor noise) with 1–3 towels at random poses. The towel fill colour is parameterised so the same generator produces the *default* and the *re-skinned* dataset with identical ground-truth boxes.

In [ ]:
DEFAULT_TOWEL_BGR = (216, 234, 240)   # #f0ead8 from shared/object_profiles.json
STRIPED_BLUE_BGR = (159, 93, 42)      # #2a5d9f from the FE 'striped' preset

def make_frame(towel_bgr, n_towels):
    img = np.zeros((480, 640, 3), np.uint8)
    for y in range(480):  # dark wooden floor gradient
        img[y, :] = (50 + y // 24, 75 + y // 20, 110 + y // 16)
    img = (img + rng.normal(0, 6, img.shape)).clip(0, 255).astype(np.uint8)
    boxes = []
    for _ in range(n_towels):
        w, h = int(rng.integers(60, 140)), int(rng.integers(40, 90))
        x, y = int(rng.integers(0, 640 - w)), int(rng.integers(160, 480 - h))
        shade = rng.integers(-12, 12, 3)
        color = tuple(int(np.clip(c + s, 0, 255)) for c, s in zip(towel_bgr, shade))
        cv2.rectangle(img, (x, y), (x + w, y + h), color, -1)
        boxes.append([x, y, x + w, y + h])
    return cv2.GaussianBlur(img, (3, 3), 0), boxes

def make_dataset(towel_bgr, n_frames=40):
    return [make_frame(towel_bgr, int(rng.integers(1, 4))) for _ in range(n_frames)]

def towel_recall(detector, dataset, iou_thr=0.3):
    hits = total = 0
    for img, gt_boxes in dataset:
        dets = [d for d in detector.detect(img) if d['class'] == 'towel']
        for gt in gt_boxes:
            total += 1
            hits += any(iou(gt, d['bbox']) >= iou_thr for d in dets)
    return hits / total

eval_default = make_dataset(DEFAULT_TOWEL_BGR)
eval_skinned = make_dataset(STRIPED_BLUE_BGR)

recall_before = towel_recall(detector, eval_default)
print(f'recall, default skin: {recall_before:.2f}')

## Re-skin without retuning: detection collapses

Blue towels fall far outside the default towel HSV band (bright, warm, low-saturation). This is the failure a data scientist sees when someone uploads a skin with `sync AI profile` unchecked.

In [ ]:
recall_skinned = towel_recall(detector, eval_skinned)
print(f'recall, blue skin, default profile: {recall_skinned:.2f}')

## Resample the profile from the skin image

`Detector.resample_profile` is the same code path as `POST /api/profiles/towel`: it derives a robust HSV band (5th–95th percentile + lighting padding) from the uploaded skin.

In [ ]:
skin_img = np.full((128, 128, 3), STRIPED_BLUE_BGR, np.uint8)
band = detector.resample_profile('towel', skin_img)
print('resampled band:', band)

recall_after = towel_recall(detector, eval_skinned)
print(f'recall, blue skin, resampled profile: {recall_after:.2f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
labels = ['default skin\ndefault profile', 'blue skin\ndefault profile', 'blue skin\nresampled profile']
values = [recall_before, recall_skinned, recall_after]
axes[0].bar(labels, values, color=['#4c9f70', '#c0504d', '#4c9f70'])
axes[0].set_ylim(0, 1); axes[0].set_ylabel('towel recall')

img, gts = eval_skinned[0]
vis = img.copy()
for d in detector.detect(img):
    x1, y1, x2, y2 = d['bbox']
    cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 255, 0), 2)
    cv2.putText(vis, f"{d['class']} {d['confidence']}", (x1, y1 - 4), 0, 0.5, (0, 255, 0), 1)
axes[1].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); axes[1].set_title('eval frame')
axes[2].imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)); axes[2].set_title('detections after resample')
for ax in axes[1:]: ax.axis('off')
plt.tight_layout()

## Cleanup + live variant

Restore the default band so later runs start clean. Against the live stack, replace the synthetic frames with `/camera/front/image_raw/compressed` frames (see notebook 01) and use ground truth from `/ground_truth/objects` to score recall on real sim imagery.

In [ ]:
detector.reset_profile('towel')
print('restored:', detector.profiles_snapshot()['towel'])